In [1]:
import glob, json, os, time
import numpy as np, pandas as pd, torch
try:
    import segmentation_models_pytorch as smp
except ImportError:
    os.system("pip install -q segmentation-models-pytorch==0.6.0")
    import segmentation_models_pytorch as smp
print("smp", smp.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
dev = "cuda"; OUT = "/kaggle/working"

def find(n):
    r = sorted(glob.glob(f"/kaggle/input/**/{n}", recursive=True)); assert r, f"not found: {n}"
    return r[0]

MODELS = {"E_consensus": 3, "E_worldcover": 3, "E_dynworld": 3, "E_consensus_1month": 1}
REFS9 = ["consensus", "worldcover", "dynworld", "esri_strict", "esri_range"]
REFS6 = ["consensus", "worldcover", "dynworld", "esri"]
CLS9 = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]
CLS6 = ["water", "trees", "low_veg", "built", "bare", "snow"]

# 9 -> 6 classes (flooded veg, grass, crops, shrub -> low vegetation; ESRI rangeland 253 -> low vegetation)
L6 = np.full(256, 255, np.int64)
for k, v in {0: 0, 1: 1, 2: 2, 3: 2, 4: 2, 5: 2, 6: 3, 7: 4, 8: 5, 253: 2}.items(): L6[k] = v
L6t = torch.tensor(L6, device=dev)

norm = json.load(open(find("s1_norm.json")))
mean = torch.tensor(norm["mean"] * 3, dtype=torch.float32, device=dev).view(1, 6, 1, 1)
std = torch.tensor(norm["std"] * 3, dtype=torch.float32, device=dev).view(1, 6, 1, 1)

nets = {}
for r, st in MODELS.items():
    m = smp.Unet("resnet18", encoder_weights=None, in_channels=6 if st == 3 else 2, classes=9).to(dev)
    m.load_state_dict(torch.load(find(f"{r}_best.pt"), map_location=dev)); m.eval(); nets[r] = m
    print("loaded", r)

Xm = np.load(find("X_test.npy"), mmap_mode="r"); Ym = np.load(find("Y_test.npy"), mmap_mode="r")
meta = np.load(find("meta_test.npz")); vi = np.where(meta["valid"])[0]
print("test patches:", len(vi), "| places:", len(set(meta["pid"][vi].tolist())))

conf9 = {(m, r): torch.zeros(9, 9, dtype=torch.long, device=dev) for m in MODELS for r in REFS9}
conf6 = {(m, r): torch.zeros(6, 6, dtype=torch.long, device=dev) for m in MODELS for r in REFS6}
def add(conf, y, p, k):
    mk = y < k
    conf += torch.bincount(y[mk] * k + p[mk], minlength=k * k).view(k, k)

t0 = time.time(); B = 64
with torch.no_grad():
    for s in range(0, len(vi), B):
        ii = vi[s:s + B]
        x = torch.from_numpy(np.asarray(Xm[ii], dtype=np.float32)).to(dev)
        x = (x - mean) / std
        y = torch.from_numpy(np.asarray(Ym[ii])).to(dev).long()
        for name, st in MODELS.items():
            with torch.autocast("cuda", dtype=torch.float16):
                out = nets[name](x if st == 3 else x[:, 2:4])
            p = out.argmax(1)
            yr = y[:, 3].clone(); rg = yr == 253
            yr[rg] = torch.where((p == 2) | (p == 5), p, torch.full_like(p, 2))[rg]
            refs = {"consensus": y[:, 0], "worldcover": y[:, 1], "dynworld": y[:, 2], "esri_strict": y[:, 3], "esri_range": yr}
            for r, yy in refs.items(): add(conf9[(name, r)], yy, p, 9)
            p6 = L6t[p]
            for r, yy in (("consensus", y[:, 0]), ("worldcover", y[:, 1]), ("dynworld", y[:, 2]), ("esri", y[:, 3])):
                add(conf6[(name, r)], L6t[yy], p6, 6)
        if (s // B) % 10 == 0: print(f"{s}/{len(vi)} {time.time() - t0:.0f}s", flush=True)

def metrics(c):
    c = c.double().cpu().numpy(); d = np.diag(c); row = c.sum(1); col = c.sum(0)
    iou = d / np.maximum(row + col - d, 1); f1 = 2 * d / np.maximum(row + col, 1); pres = row > 0
    return d.sum() / c.sum(), iou[pres].mean(), f1[pres].mean(), iou, f1

def table(conf, refs, cls):
    rows = []
    for (m, r), c in conf.items():
        oa, miou, mf1, iou, f1 = metrics(c)
        row = dict(model=m, ref=r, OA=oa, mIoU=miou, mF1=mf1)
        row.update({f"IoU_{n}": v for n, v in zip(cls, iou)}); row.update({f"F1_{n}": v for n, v in zip(cls, f1)})
        rows.append(row)
    return pd.DataFrame(rows)

t9 = table(conf9, REFS9, CLS9); t6 = table(conf6, REFS6, CLS6)
t9.to_csv(f"{OUT}/test_results_9class.csv", index=False); t6.to_csv(f"{OUT}/test_results_6class.csv", index=False)
np.savez(f"{OUT}/test_confusions.npz", **{f"c9__{m}__{r}": c.cpu().numpy() for (m, r), c in conf9.items()},
         **{f"c6__{m}__{r}": c.cpu().numpy() for (m, r), c in conf6.items()})

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
for r in REFS9:
    print(f"\n=== 9 classes | reference: {r} ===")
    sub = t9[t9.ref == r].set_index("model")
    print(sub[["OA", "mIoU", "mF1"] + [f"IoU_{n}" for n in CLS9]].round(3).to_string())
for r in REFS6:
    print(f"\n=== 6 classes | reference: {r} ===")
    sub = t6[t6.ref == r].set_index("model")
    print(sub[["OA", "mIoU", "mF1"] + [f"IoU_{n}" for n in CLS6]].round(3).to_string())

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, r in zip(ax, ["consensus", "esri_range"]):
    c = conf9[("E_consensus", r)].double().cpu().numpy(); c = c / np.maximum(c.sum(1, keepdims=True), 1)
    a.imshow(c, vmin=0, vmax=1, cmap="Blues"); a.set_title(f"E_consensus vs {r} (row-normalised)")
    a.set_xticks(range(9)); a.set_xticklabels(CLS9, rotation=45); a.set_yticks(range(9)); a.set_yticklabels(CLS9)
    a.set_xlabel("predicted"); a.set_ylabel("reference")
    for i in range(9):
        for j in range(9): a.text(j, i, f"{c[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/test_confusion_E_consensus.png", dpi=120)
print("\nfiles:", sorted(os.listdir(OUT)))

smp 0.6.0 | torch 2.11.0+cu130 | cuda True
loaded E_consensus
loaded E_worldcover
loaded E_dynworld
loaded E_consensus_1month
test patches: 4402 | places: 147
0/4402 8s
640/4402 18s
1280/4402 28s
1920/4402 38s
2560/4402 48s
3200/4402 60s
3840/4402 77s

=== 9 classes | reference: consensus ===
                       OA   mIoU    mF1  IoU_water  IoU_trees  IoU_grass  IoU_flooded  IoU_crops  IoU_shrub  IoU_built  IoU_bare  IoU_snow
model                                                                                                                                     
E_consensus         0.778  0.586  0.721      0.821      0.780      0.414        0.408      0.637      0.343      0.795     0.706     0.372
E_worldcover        0.758  0.555  0.691      0.830      0.753      0.394        0.329      0.607      0.283      0.756     0.703     0.343
E_dynworld          0.638  0.456  0.582      0.723      0.628      0.118        0.299      0.539      0.309      0.793     0.618     0.074
E_consensus